In [ ]:
from langgraph.graph import StateGraph , START, END
from typing import TypedDict, Literal
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llm = ChatGoogleGenerativeAI(
    Model = "",
    key = ""
)


class ProductState(TypedDict):
    review : str
    sentiment : str
    response : str


def find_sentiment(state:ProductState)->ProductState:
   query = state['review']
   prompt = f" find the sentiment just return only positive or negative based on the review:{query}"
   sentiment =llm.invoke(prompt).content
   state['sentiment']= sentiment.lower().strip()
   return state


In [ ]:
def generate_Neg_response(state:ProductState)->ProductState:
   query = state['review'] 
   prompt1 = f" As an Assistant return the tone of the given query :{query}"
   prompt2 =  f" As an Assistant detect the urgency in the given query :{query}"
   prompt3 = f" As an Assistant find out the issue in the given negative review :{query}"

   tone = llm.invoke(prompt1).content
   urgency = llm.invoke(prompt2).content
   issue = llm.invoke(prompt3).content

   res_prompt = f"As a research assistant generate the proffessional response of negative review on the basis of three things urgency:{urgency} ,issue_type :{issue} , tone: {tone}"
   state['response'] =llm.invoke(res_prompt).content
   return state


In [ ]:

def generate_Pos_response(state:ProductState)->ProductState:
   query = state['review']
   prompt = f" As a Company Product Dealer Generate the positive response on the Basis of Given query:{query}"
   response = llm.invoke(prompt).content
   state['response'] = response
   return state



def check_condition(state:ProductState)->Literal["Positive","Negative"]:
   if state['sentiment']=="positive":
      return "Positive"
   else :
      return "Negative"

In [ ]:
graph = StateGraph(ProductState)

graph.add_node('Sentiment',find_sentiment)
graph.add_node('Positive',generate_Pos_response)
graph.add_node('Negative',generate_Neg_response)



graph.add_edge(START,"Sentiment")
graph.add_conditional_edges("Sentiment",check_condition)
graph.add_edge("Positive",END)
graph.add_edge("Negative",END)

workflow =graph.compile()
